# Formatting changes to reference pathway evaluation table - April 13, 2018
Rename the pathways, add a derived column, filter the algorithms.  Based on FormatReferenceEval081417.ipynb but the columns and their order have changed.

In [1]:
%matplotlib inline
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import os

eval_dir = os.path.join('..','..','evaluation','2018-03-02-randomized-network-reference-comparison')
in_file = os.path.join(eval_dir, 'aggregate-randomized-reference-comparison.csv')
out_file = os.path.join(eval_dir, 'aggregate-randomized-reference-comparison-formatted.tsv')

## Read the input table
Use different header names

In [2]:
header = ['Method original', 'Reference', 'Network size', 'Reference size', 'Undirected edge matches',
          'Undirected precision', 'Undirected recall', 'Directed predictions', 'Directed edge ratio', 'Directed edge matches',
          'Directed edge match ratio', 'Correct directions', 'Correct directions ratio', 'Incorrect directions',
          'Incorrect directions ratio', 'Unevaluated directions', 'Unevaluated directions ratio']
eval_df = pd.read_csv(in_file, sep=',', header = 0, names = header)
print 'Loaded %d evaluation rows' % len(eval_df)
assert len(eval_df) == 12
eval_df.head()

Loaded 12 evaluation rows


,Method original,Reference,Network size,Reference size,Undirected edge matches,Undirected precision,Undirected recall,Directed predictions,Directed edge ratio,Directed edge matches,Directed edge match ratio,Correct directions,Correct directions ratio,Incorrect directions,Incorrect directions ratio,Unevaluated directions,Unevaluated directions ratio
0,aggregate-results,biocarta-edges.tsv,372,1750,1,0.003811,0.000840,251,0.669798,1,0.003953,0,0.0,0,0,0,0.000000
1,aggregate-results,biocarta-egfr-edges.tsv,372,18,0,0.000290,0.005556,251,0.669798,0,0.000000,0,0.0,0,0,0,0.000000
2,aggregate-results,cellmap-commons-egfr-edges.tsv,372,318,2,0.005230,0.006572,251,0.669798,1,0.004587,0,0.0,0,0,1,0.003731
3,aggregate-results,kegg-4010-edges.tsv,372,122,0,0.000728,0.002377,251,0.669798,0,0.000000,0,0.0,0,0,0,0.000000
4,aggregate-results,kegg-4012-edges.tsv,372,54,0,0.000487,0.003519,251,0.669798,0,0.000000,0,0.0,0,0,0,0.000000


## Test derived columns
These values are the median from the distribution of runs so not all conditions below will hold.

In [3]:
for (index, row) in eval_df.iterrows():
    #assert row['Directed edge matches'] == row['Correct directions'] + row['Incorrect directions'] + row['Unevaluated directions']
    #assert abs(row['Undirected precision'] - (float(row['Undirected edge matches'])/row['Network size'])) < 0.0001, row
    #assert abs(row['Undirected recall'] - (float(row['Undirected edge matches'])/row['Reference size'])) < 0.0001, row
    assert row['Directed predictions'] <= row['Network size']

## Rename pathways, rename methods, ignore methods
Drop the Panther evaluation.  We did not use it when evaluating phosphorylation overlap and it does not add any new information to the comparison.

Use the same pathway names used in the phoshorylation overlap figures.  Rename the method.

Drop the columns that are not used when evaluating real networks and reorder the remaining columns.

In [4]:
# Map the method names to human-readable names and drop all methods not in this map
method_names = {'aggregate-results':'TPS with randomized network'}
eval_df = eval_df.loc[eval_df['Method original'].isin(method_names.keys())]
print '%d methods' % len(method_names)

# Rename the method column
eval_df.insert(0, 'Method', eval_df.apply(lambda row: method_names[row['Method original']], axis=1))
eval_df = eval_df.drop('Method original', axis=1)

# Remove the Panther rows
eval_df = eval_df.loc[eval_df['Reference'] != 'panther-edges.tsv']

# Map the pathway names to human-readable names
pathway_names = {'kegg-4012':'KEGG ErbB', # http://www.genome.jp/kegg-bin/show_pathway?hsa04012
                    'kegg-4010':'KEGG MAPK', # http://www.genome.jp/kegg-bin/show_pathway?hsa04010
                    'layek':'Layek et al.', # http://bioinformatics.oxfordjournals.org/content/27/4/548/F4.large.jpg
                    'reactome-reactome-egfr':'Reactome EGFR',
                    'biocarta-egfr':'BioCarta EGF', # http://www.biocarta.com/pathfiles/h_egfpathway.asp
                    'science-signaling-egfr':'Science Signaling EGFR', # Not longer available online
                    'cellmap-commons-egfr':'Cancer Cell Map EGFR1', # http://www.pathwaycommons.org/pc/dbSnapshot.do?snapshot_id=8
                    'nci-nature-egfr':'PID ErbB1', # http://pid.nci.nih.gov/search/pathway_landing.shtml?pathway_id=200101&source=NATURE&what=graphic&jpg=on
                    'biocarta':'All BioCarta',
                    'reactome-reactome':'All Reactome',
                    'nci-nature':'All PID'}
print '%d reference pathways' % len(pathway_names)

# Rename the reference pathway column
eval_df.insert(0, 'Reference pathway', eval_df.apply(lambda row: pathway_names[row['Reference'].replace('-edges.tsv', '')], axis=1))
eval_df = eval_df.drop('Reference', axis=1)

# Confirm the expected number of rows remain
assert len(eval_df) == len(method_names) * len(pathway_names)

# Remove columns that were not in the earlier version of the output
eval_df = eval_df.drop('Directed edge ratio', axis=1)
eval_df = eval_df.drop('Directed edge match ratio', axis=1)
eval_df = eval_df.drop('Correct directions ratio', axis=1)
eval_df = eval_df.drop('Incorrect directions ratio', axis=1)
eval_df = eval_df.drop('Unevaluated directions ratio', axis=1)

# Reorder the columns
columns = ['Reference pathway', 'Method', 'Reference size', 'Network size', 'Undirected edge matches',
          'Undirected precision', 'Undirected recall', 'Directed predictions', 'Directed edge matches',
          'Correct directions', 'Incorrect directions', 'Unevaluated directions']
eval_df = eval_df[columns]

eval_df.head()

1 methods
11 reference pathways


,Reference pathway,Method,Reference size,Network size,Undirected edge matches,Undirected precision,Undirected recall,Directed predictions,Directed edge matches,Correct directions,Incorrect directions,Unevaluated directions
0,All BioCarta,TPS with randomized network,1750,372,1,0.003811,0.000840,251,1,0,0,0
1,BioCarta EGF,TPS with randomized network,18,372,0,0.000290,0.005556,251,0,0,0,0
2,Cancer Cell Map EGFR1,TPS with randomized network,318,372,2,0.005230,0.006572,251,1,0,0,1
3,KEGG MAPK,TPS with randomized network,122,372,0,0.000728,0.002377,251,0,0,0,0
4,KEGG ErbB,TPS with randomized network,54,372,0,0.000487,0.003519,251,0,0,0,0


## Helper function for directed precision

In [5]:
def precision(correct, incorrect):
    total = correct + incorrect
    if total == 0:
        return 'N/A'
    else:
        return float(correct)/total

##  Add derived columns
Normalize the number of confirmed and wrong orientations by the number of oriented predictions.  This makes it easier to see the rate at which different methods make correct and incorrect direction predictions.  Add the directed precision.  It is hard to determine directed recall.  Directed precision does not make as much sense when calculated on the median values here, but keep it for consistency.

In [6]:
eval_df['Directed precision'] = eval_df.apply(lambda row: precision(row['Correct directions'], row['Incorrect directions']), axis=1)
eval_df['Correct directions per 1000'] = eval_df.apply(lambda row: float(row['Correct directions'])*1000/row['Directed predictions'], axis=1)
eval_df['Incorrect directions per 1000'] = eval_df.apply(lambda row: float(row['Incorrect directions'])*1000/row['Directed predictions'], axis=1)

eval_df.head()

,Reference pathway,Method,Reference size,Network size,Undirected edge matches,Undirected precision,Undirected recall,Directed predictions,Directed edge matches,Correct directions,Incorrect directions,Unevaluated directions,Directed precision,Correct directions per 1000,Incorrect directions per 1000
0,All BioCarta,TPS with randomized network,1750,372,1,0.003811,0.000840,251,1,0,0,0,N/A,0.0,0.0
1,BioCarta EGF,TPS with randomized network,18,372,0,0.000290,0.005556,251,0,0,0,0,N/A,0.0,0.0
2,Cancer Cell Map EGFR1,TPS with randomized network,318,372,2,0.005230,0.006572,251,1,0,0,1,N/A,0.0,0.0
3,KEGG MAPK,TPS with randomized network,122,372,0,0.000728,0.002377,251,0,0,0,0,N/A,0.0,0.0
4,KEGG ErbB,TPS with randomized network,54,372,0,0.000487,0.003519,251,0,0,0,0,N/A,0.0,0.0


## Sort the table rows
Smallest reference pathways first.  TPS first.  Sorting is case-sensitive so temporarily add a new column for sorting the methods.

In [7]:
eval_df['Method sort'] = eval_df.apply(lambda row: row['Method'].upper(), axis=1)
eval_df = eval_df.sort(['Reference size','Method sort'], ascending=[True,False])
eval_df = eval_df.drop('Method sort', axis=1)
eval_df.head()

C:\Program Files\Anaconda\lib\site-packages\IPython\kernel\__main__.py:2: FutureWarning: sort(columns=....) is deprecated, use sort_values(by=.....)
  from IPython.kernel.zmq import kernelapp as app


,Reference pathway,Method,Reference size,Network size,Undirected edge matches,Undirected precision,Undirected recall,Directed predictions,Directed edge matches,Correct directions,Incorrect directions,Unevaluated directions,Directed precision,Correct directions per 1000,Incorrect directions per 1000
1,BioCarta EGF,TPS with randomized network,18,372,0,0.000290,0.005556,251,0,0,0,0,N/A,0.0,0.0
7,PID ErbB1,TPS with randomized network,44,372,0,0.001383,0.012045,251,0,0,0,0,N/A,0.0,0.0
4,KEGG ErbB,TPS with randomized network,54,372,0,0.000487,0.003519,251,0,0,0,0,N/A,0.0,0.0
10,Reactome EGFR,TPS with randomized network,80,372,0,0.001074,0.005500,251,0,0,0,0,N/A,0.0,0.0
5,Layek et al.,TPS with randomized network,83,372,0,0.000805,0.003735,251,0,0,0,0,N/A,0.0,0.0


## Write the updated table

In [8]:
eval_df.to_csv(out_file, sep='\t', index=False)
print 'Wrote %s' % out_file

Wrote ..\..\evaluation\2018-03-02-randomized-network-reference-comparison\aggregate-randomized-reference-comparison-formatted.tsv
